# 02: API Error Handling & Multiple Cities

## What You Will Learn
- Why error handling is critical
- How to write robust API functions
- How to handle different error types
- How to fetch multiple cities in a loop
- How to collect and clean data

---

## Why Error Handling Matters

Real-world APIs fail. A lot. Reasons:
- ❌ City doesn't exist (404)
- ❌ Network timeout (no internet)
- ❌ API key expired (401)
- ❌ Rate limit hit (429)
- ❌ Server crash (500)

**Without handling:** Your script crashes and stops.
**With handling:** Your script continues, skips bad data, logs issues.

## Step 1: Setup - Same as Before

In [1]:
import requests
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENWEATHER_API_KEY")

print(f"API Key loaded: {api_key[:5]}..." if api_key else "ERROR: Key not found")

API Key loaded: a71d7...


## Step 2: Create a Robust API Function

**Features of this function:**
- `try/except` → Catches network errors
- `timeout=10` → Won't hang forever
- Status code checks → Specific error messages
- Returns `None` on failure → Caller can check
- Docstring → Explains what it does

In [2]:
def get_weather(city_name):
    """
    Fetch weather data for a city with error handling.
    
    Args:
        city_name (str): Name of the city
    
    Returns:
        dict: Weather data if successful, None if failed
    """
    # Build the API URL
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={api_key}&units=metric"
    
    try:
        # Send request with 10-second timeout
        response = requests.get(url, timeout=10)
        
        # Check status code and return appropriate result
        if response.status_code == 200:
            return response.json()  # Success: return data
        elif response.status_code == 404:
            print(f"  ✗ Error: City '{city_name}' not found")
            return None
        elif response.status_code == 401:
            print(f"  ✗ Error: Invalid API key")
            return None
        else:
            print(f"  ✗ Error: API returned status {response.status_code}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"  ✗ Error: Request timed out for '{city_name}'")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Network error: {e}")
        return None

## Step 3: Test with Single City

In [3]:
# Test with a real city
print("Testing with real city:")
weather = get_weather("Delhi")

if weather:
    print(f"  ✓ Success: {weather['name']} is {weather['main']['temp']}°C")
else:
    print("  ✗ Failed to get weather")

Testing with real city:
  ✓ Success: Delhi is 33.05°C


In [4]:
# Test with fake city (should fail gracefully)
print("\nTesting with fake city:")
weather = get_weather("XyzFakeCity123")

if weather:
    print(f"  ✓ Success")
else:
    print("  ✗ As expected, failed gracefully")


Testing with fake city:
  ✗ Error: City 'XyzFakeCity123' not found
  ✗ As expected, failed gracefully


## Step 4: Fetch Multiple Cities

**Pattern:**
1. Create list of cities
2. Loop through each city
3. Call `get_weather()` for each
4. If success → extract clean data → add to list
5. If fail → skip (don't add None to list)
6. End up with clean list of successful results

In [5]:
# Define cities to fetch
cities = ["Delhi", "Mumbai", "London", "XyzFakeCity", "Tokyo", "New York"]

# Empty list to store successful results
all_weather_data = []

print("Fetching weather for multiple cities...\n")

# Loop through each city
for city in cities:
    print(f"Fetching: {city}")
    
    # Call our safe function
    weather = get_weather(city)
    
    # Only process if we got valid data (not None)
    if weather:
        # Extract only the fields we need
        clean_data = {
            'city': weather['name'],
            'country': weather['sys']['country'],
            'temperature': weather['main']['temp'],
            'feels_like': weather['main']['feels_like'],
            'humidity': weather['main']['humidity'],
            'pressure': weather['main']['pressure'],
            'description': weather['weather'][0]['description'],
            'wind_speed': weather['wind']['speed']
        }
        all_weather_data.append(clean_data)
        print(f"  ✓ Got data: {clean_data['city']} - {clean_data['temperature']}°C")
    else:
        print(f"  ✗ Skipped (API call failed)")

print(f"\n=== Summary ===")
print(f"Successfully fetched: {len(all_weather_data)} out of {len(cities)} cities")

Fetching weather for multiple cities...

Fetching: Delhi
  ✓ Got data: Delhi - 33.05°C
Fetching: Mumbai
  ✓ Got data: Mumbai - 32.99°C
Fetching: London
  ✓ Got data: London - 8.97°C
Fetching: XyzFakeCity
  ✗ Error: City 'XyzFakeCity' not found
  ✗ Skipped (API call failed)
Fetching: Tokyo
  ✓ Got data: Tokyo - 21.78°C
Fetching: New York
  ✓ Got data: New York - 14.23°C

=== Summary ===
Successfully fetched: 5 out of 6 cities


## Step 5: View the Collected Data

In [6]:
# Look at the clean data we collected
print("\nRaw data structure (list of dictionaries):\n")
print(all_weather_data)

print("\n\nFormatted view:")
for record in all_weather_data:
    print(f"\n{record['city']}, {record['country']}:")
    print(f"  Temp: {record['temperature']}°C (feels like {record['feels_like']}°C)")
    print(f"  Humidity: {record['humidity']}% | Wind: {record['wind_speed']} m/s")
    print(f"  Condition: {record['description']}")


Raw data structure (list of dictionaries):

[{'city': 'Delhi', 'country': 'IN', 'temperature': 33.05, 'feels_like': 32.53, 'humidity': 33, 'pressure': 1008, 'description': 'haze', 'wind_speed': 3.6}, {'city': 'Mumbai', 'country': 'IN', 'temperature': 32.99, 'feels_like': 36.86, 'humidity': 52, 'pressure': 1010, 'description': 'smoke', 'wind_speed': 3.09}, {'city': 'London', 'country': 'GB', 'temperature': 8.97, 'feels_like': 8.97, 'humidity': 87, 'pressure': 1020, 'description': 'broken clouds', 'wind_speed': 0.89}, {'city': 'Tokyo', 'country': 'JP', 'temperature': 21.78, 'feels_like': 21.32, 'humidity': 50, 'pressure': 1015, 'description': 'broken clouds', 'wind_speed': 6.69}, {'city': 'New York', 'country': 'US', 'temperature': 14.23, 'feels_like': 13.98, 'humidity': 87, 'pressure': 1016, 'description': 'clear sky', 'wind_speed': 3.6}]


Formatted view:

Delhi, IN:
  Temp: 33.05°C (feels like 32.53°C)
  Humidity: 33% | Wind: 3.6 m/s
  Condition: haze

Mumbai, IN:
  Temp: 32.99°C (fe

---

## Key Takeaways

✅ **Always wrap API calls in try/except**
✅ **Check status codes before processing data**
✅ **Return None on failure, let caller decide what to do**
✅ **Use loops to process multiple items**
✅ **Extract only fields you need → cleaner data**
✅ **Failed requests shouldn't stop your entire script**

**Next:** `03_Pandas_DataFrame.ipynb` → Convert this list into a proper DataFrame